We're building a custom dataset by combining ETF options chains from yfinance with earnings dates for each ETF's top holdings, scraped from stockanalysis.com. We verified scraping is allowed via their robots.txt. Our main question is whether implied volatility in ETF options responds to earnings events from the ETF's largest individual holdings

# Section 1 — Polite & Robust Scraping Compliance Check

In [ ]:
# Imports

import time
import logging
import requests
from urllib.robotparser import RobotFileParser
from urllib.parse import urljoin

In [ ]:
# Shared scraping configuration

# Single source of truth for headers and delay used across
# all scraping sections of this notebook.

BASE_URL   = "https://stockanalysis.com"
ROBOTS_URL = urljoin(BASE_URL, "/robots.txt")
DELAY      = 1.5   # seconds between consecutive requests

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger(__name__)

print(f"User-Agent : {HEADERS['User-Agent']}")
print(f"Delay      : {DELAY}s between requests")
print(f"Robots URL : {ROBOTS_URL}")

In [ ]:
# Fetch and display robots.txt

try:
    resp = requests.get(ROBOTS_URL, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    robots_text = resp.text
    print(f"✓ robots.txt fetched ({len(robots_text)} chars)\n")
    print("=" * 50)
    print(robots_text)
    print("=" * 50)
except requests.RequestException as e:
    robots_text = ""
    print(f"⚠ Could not fetch robots.txt: {e}")
    print("Proceeding with caution — assuming no restrictions.")

In [ ]:
# Parse robots.txt and check our URLs

URLS_TO_CHECK = [
    "/etf/qqq/holdings/",
    "/etf/voo/holdings/",
    "/etf/arkq/holdings/",
    "/etf/botz/holdings/",
    "/stocks/aapl/statistics/",
    "/stocks/nvda/statistics/",
]

# ── Detect blank Disallow (means "allow everything") ────────
# Python's RobotFileParser misinterprets a blank Disallow: line
# as "block all" when it actually means "block nothing."
# We check for this manually before falling back to the parser.

blank_disallow = (
    "User-agent: *\nDisallow:\n" in robots_text
    or "User-agent: *\nDisallow: \n" in robots_text
)

if blank_disallow:
    # Blank Disallow = no restrictions for all crawlers
    print("✓ robots.txt has blank Disallow for User-agent: *")
    print("  This means ALL paths are permitted — no restrictions.\n")

    print(f"{'URL':<45} {'Allowed?'}")
    print("-" * 55)
    for path in URLS_TO_CHECK:
        print(f"{path:<45} ✓ Allowed")

    print()
    print("✓ All scraped URLs are permitted by robots.txt")

else:
    # Fall back to RobotFileParser for normal robots.txt rules
    rp = RobotFileParser()
    rp.set_url(ROBOTS_URL)

    try:
        rp.read()
        print("✓ robots.txt parsed successfully\n")
    except Exception as e:
        print(f"⚠ Could not parse robots.txt: {e}\n")

    print(f"{'URL':<45} {'Allowed?'}")
    print("-" * 55)

    all_allowed = True
    for path in URLS_TO_CHECK:
        full_url = urljoin(BASE_URL, path)
        allowed  = rp.can_fetch("*", full_url)
        status   = "✓ Allowed" if allowed else "✗ DISALLOWED"
        if not allowed:
            all_allowed = False
        print(f"{path:<45} {status}")

    print()
    if all_allowed:
        print("✓ All scraped URLs are permitted by robots.txt")
    else:
        print("⚠ Some URLs are disallowed — review before scraping")

In [ ]:
# Demonstrate robust request wrapper

def polite_get(url: str, delay: float = DELAY) -> requests.Response | None:
    """
    Perform a single GET request with:
      - Descriptive User-Agent header
      - Timeout to avoid hanging indefinitely
      - try/except for all HTTP and connection errors
      - Logged warning on failure instead of crash
      - Polite delay after every request

    Parameters
    ----------
    url   : Full URL to fetch.
    delay : Seconds to sleep after the request (default: DELAY).

    Returns
    -------
    requests.Response if successful, None otherwise.
    """
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()   # raises for 4xx / 5xx
        return resp

    except requests.exceptions.Timeout:
        log.warning("Timeout fetching %s — skipping.", url)
    except requests.exceptions.ConnectionError:
        log.warning("Connection error fetching %s — skipping.", url)
    except requests.exceptions.HTTPError as e:
        status = e.response.status_code
        if status == 404:
            log.warning("404 Not Found: %s — page does not exist, skipping.", url)
        elif status == 429:
            log.warning("429 Too Many Requests: %s — slow down.", url)
        elif 400 <= status < 500:
            log.warning("Client error %d for %s — skipping.", status, url)
        elif 500 <= status < 600:
            log.warning("Server error %d for %s — skipping.", status, url)
    except requests.exceptions.RequestException as e:
        log.warning("Unexpected request error for %s: %s — skipping.", url, e)
    finally:
        time.sleep(delay)   # always delay, even on failure

    return None


In [ ]:
# Live test of polite_get on known URLs

test_urls = [
    "https://stockanalysis.com/etf/qqq/holdings/",   # should succeed
    "https://stockanalysis.com/stocks/aapl/statistics/",  # should succeed
    "https://stockanalysis.com/stocks/tlv:%20eslt/statistics/",  # 404 — known bad
]

print("Testing polite_get on sample URLs:\n")
for url in test_urls:
    print(f"→ {url}")
    resp = polite_get(url)
    if resp:
        print(f"  ✓ Status {resp.status_code} — {len(resp.text):,} chars received\n")
    else:
        print(f"  ✗ Skipped gracefully\n")

In [ ]:
# Compliance summary

print("=" * 55)
print("POLITE SCRAPING COMPLIANCE SUMMARY")
print("=" * 55)
print(f"  User-Agent header    : ✓ Set on all requests")
print(f"  Request delay        : ✓ {DELAY}s between each request")
print(f"  Error handling       : ✓ try/except for 4xx, 5xx,")
print(f"                             timeouts, connection errors")
print(f"  Failure behavior     : ✓ Logs warning, skips, no crash")
print(f"  robots.txt checked   : ✓ All scraped URLs permitted")
print(f"  Pages scraped        : /etf/*/holdings/")
print(f"                         /stocks/*/statistics/")
print("=" * 55)

# Section 1 — Price History

The `etf_iv` package ships a bundled dataset so you can run the analysis without any API calls. The cell below loads it directly. If you want **fresh data**, skip the load cell and run the "Optional" collection cells that follow.

In [ ]:
# Load bundled price history (no network needed)

from etf_iv.data import load_price_history

prices = load_price_history()
print(f"Loaded bundled price history: {len(prices):,} rows, "
      f"{prices['ticker'].nunique()} tickers")
prices.head()

### Optional: Collect Fresh Price History

Skip these cells if you loaded the bundled data above. Run them to fetch **current** price data from yfinance.

In [ ]:
# Imports

import pandas as pd
from etf_iv import get_price_history

In [ ]:
# Configuration

TICKERS = ["VOO", "QQQ", "ARKQ", "BOTZ"]
PERIOD  = "6mo"
OUTPUT_CSV = "price_history.csv"

In [ ]:
# Fetch price history for all tickers

frames = []

for ticker in TICKERS:
    print(f"Fetching price history: {ticker} ...", end=" ")
    try:
        df = get_price_history(ticker, period=PERIOD)
        frames.append(df)
        print(f"✓  {len(df):,} rows")
    except ValueError as e:
        print(f"✗  Skipped — {e}")

if not frames:
    raise RuntimeError("No data collected. Check ticker symbols and network.")

prices = pd.concat(frames, ignore_index=False)

In [ ]:
# Clean and standardize

# Reset index so Date becomes a regular column
prices = prices.reset_index()

# Rename to match acceptance criteria exactly
prices = prices.rename(columns={"index": "Date", "ticker": "ticker"})

# Keep only required columns
KEEP_COLS = ["ticker", "Date", "Open", "High", "Low", "Close", "Volume"]
cols = [c for c in KEEP_COLS if c in prices.columns]
prices = prices[cols]

# Ensure correct dtypes
prices["Date"]   = pd.to_datetime(prices["Date"]).dt.tz_localize(None)
prices["Open"]   = pd.to_numeric(prices["Open"],   errors="coerce")
prices["High"]   = pd.to_numeric(prices["High"],   errors="coerce")
prices["Low"]    = pd.to_numeric(prices["Low"],    errors="coerce")
prices["Close"]  = pd.to_numeric(prices["Close"],  errors="coerce")
prices["Volume"] = pd.to_numeric(prices["Volume"], errors="coerce")

# Sort for readability
prices = prices.sort_values(["ticker", "Date"]).reset_index(drop=True)

prices.head(10)


In [ ]:
# Verify no major gaps in the time series

print("=== Gap Analysis ===\n")

for ticker in TICKERS:
    subset = prices[prices["ticker"] == ticker].copy()

    if subset.empty:
        print(f"{ticker}: No data.\n")
        continue

    # Expected trading days (roughly 5 per week, minus holidays)
    date_range = pd.bdate_range(subset["Date"].min(), subset["Date"].max())
    expected   = len(date_range)
    actual     = len(subset)
    missing    = expected - actual

    # Find specific gap dates
    actual_dates   = set(subset["Date"].dt.date)
    expected_dates = set(date_range.date)
    gap_dates      = sorted(expected_dates - actual_dates)

    print(f"{ticker}:")
    print(f"  Date range : {subset['Date'].min().date()} → {subset['Date'].max().date()}")
    print(f"  Expected   : {expected} trading days")
    print(f"  Actual     : {actual} rows")
    print(f"  Missing    : {missing} days", end="")

    if missing == 0:
        print(" ✓ No gaps")
    elif missing <= 5:
        print(f" — likely holidays: {gap_dates}")
    else:
        print(f" ⚠ Potential data issue — gap dates: {gap_dates[:10]}")
    print()

In [ ]:
# Quick summary stats

summary = (
    prices.groupby("ticker")
    .agg(
        rows        = ("Close", "count"),
        start_date  = ("Date",  "min"),
        end_date    = ("Date",  "max"),
        avg_close   = ("Close", "mean"),
        min_close   = ("Close", "min"),
        max_close   = ("Close", "max"),
        avg_volume  = ("Volume","mean"),
    )
    .round(2)
)
print(summary)


In [ ]:
# Save to CSV

prices.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved CSV → {OUTPUT_CSV}  ({prices.shape[0]:,} rows, {prices.shape[1]} columns)")


In [ ]:
# Reload check

reloaded = pd.read_csv(OUTPUT_CSV, parse_dates=["Date"])
assert reloaded.shape == prices.shape, "Reload shape mismatch!"
print(f"Reload check passed — {reloaded.shape[0]:,} rows, {reloaded.shape[1]} columns.")
reloaded.head()

# Section 2 — Options Chain

Load the bundled options chain below, or skip to the "Optional" cells to fetch current data from yfinance.

In [ ]:
# Load bundled options chain (no network needed)

from etf_iv.data import load_options_chain

options = load_options_chain()
print(f"Loaded bundled options chain: {len(options):,} rows, "
      f"{options['ticker'].nunique()} tickers")
options.head()

### Optional: Collect Fresh Options Chain

Skip these cells if you loaded the bundled data above. Run them to fetch **current** options data from yfinance.

In [ ]:
# Imports 

import pandas as pd
from etf_iv import get_options_chain

In [ ]:
# Configuration

TICKERS = ["VOO", "QQQ", "ARKQ", "BOTZ"]

KEEP_COLS = [
    "ticker",
    "expiration",
    "contractType",   # renamed from option_type for clarity
    "strike",
    "bid",
    "ask",
    "volume",
    "openInterest",
    "impliedVolatility",
]

OUTPUT_CSV     = "options_chain.csv"

In [ ]:
# Fetch options chain for all tickers

frames = []

for ticker in TICKERS:
    print(f"Fetching options chain: {ticker} ...", end=" ")
    try:
        df = get_options_chain(ticker)

        # Rename option_type -> contractType to match acceptance criteria
        df = df.rename(columns={"option_type": "contractType"})

        # Keep only the required columns (drop any that don't exist gracefully)
        cols = [c for c in KEEP_COLS if c in df.columns]
        df = df[cols]

        frames.append(df)
        print(f"✓  {len(df):,} rows")

    except ValueError as e:
        print(f"✗  Skipped — {e}")

if not frames:
    raise RuntimeError("No data collected. Check ticker symbols and network.")

options = pd.concat(frames, ignore_index=True)
print(f"\nTotal rows collected: {len(options):,}")

In [ ]:
# Light cleaning

# Ensure correct dtypes
options["expiration"]       = pd.to_datetime(options["expiration"])
options["strike"]           = pd.to_numeric(options["strike"],           errors="coerce")
options["bid"]              = pd.to_numeric(options["bid"],              errors="coerce")
options["ask"]              = pd.to_numeric(options["ask"],              errors="coerce")
options["volume"]           = pd.to_numeric(options["volume"],           errors="coerce")
options["openInterest"]     = pd.to_numeric(options["openInterest"],     errors="coerce")
options["impliedVolatility"]= pd.to_numeric(options["impliedVolatility"],errors="coerce")

# Drop rows where strike or IV is missing — not useful for analysis
before = len(options)
options = options.dropna(subset=["strike", "impliedVolatility"])
dropped = before - len(options)
if dropped:
    print(f"Dropped {dropped:,} rows with missing strike or impliedVolatility.")

# Sort for readability
options = options.sort_values(
    ["ticker", "expiration", "contractType", "strike"]
).reset_index(drop=True)

options.head(10)


In [ ]:
# Quick summary

summary = (
    options.groupby(["ticker", "contractType"])
    .agg(
        expirations   = ("expiration", "nunique"),
        contracts     = ("strike",     "count"),
        avg_iv        = ("impliedVolatility", "mean"),
        total_oi      = ("openInterest", "sum"),
    )
    .round(4)
)
print(summary)

In [ ]:
# Save to disk

options.to_csv(OUTPUT_CSV, index=False)
print(f"Saved CSV     → {OUTPUT_CSV}  ({options.shape[0]:,} rows)")

# Section 3 — Spot Price Join

The bundled dataset already includes options with spot prices and moneyness. Load it below, or run the "Optional" cells to recompute from fresh data.

In [ ]:
# Load bundled options with spot price and moneyness (no network needed)

from etf_iv.data import load_options_filtered

options_with_spot = load_options_filtered()
print(f"Loaded bundled options with spot: {len(options_with_spot):,} rows")
options_with_spot.head()

### Optional: Recompute Spot Price Join

Skip these cells if you loaded the bundled data above.

In [ ]:
# Load saved data

options = pd.read_csv("options_chain.csv", parse_dates=["expiration"])
prices  = pd.read_csv("price_history.csv", parse_dates=["Date"])

print(f"Options rows : {len(options):,}")
print(f"Price rows   : {len(prices):,}")

In [ ]:
# Build spot price lookup table

# We want the most recent closing price for each ticker as of today (the snapshot date when data was collected).
# Since price history is sorted oldest -> newest, the last row per ticker is the most recent closing price.

spot = (
    prices.sort_values("Date")
    .groupby("ticker", as_index=False)
    .last()                          # most recent row per ticker
    [["ticker", "Date", "Close"]]
    .rename(columns={
        "Close": "spot_price",
        "Date" : "spot_date",
    })
)

print("\nSpot prices used for join:")
print(spot.to_string(index=False))

In [ ]:
# Join spot price onto options chain

# Simple left join on ticker — every option row gets the spot price of its underlying ETF on the collection date.

options_with_spot = options.merge(spot[["ticker", "spot_price"]], 
                                   on="ticker", 
                                   how="left")

print(f"\nRows before join : {len(options):,}")
print(f"Rows after join  : {len(options_with_spot):,}")

In [ ]:
# Verify no unexpected NaNs in spot_price

nan_count = options_with_spot["spot_price"].isna().sum()

if nan_count == 0:
    print("✓ No NaNs in spot_price — join successful.")
else:
    # Show which tickers failed to join
    problem_tickers = (
        options_with_spot[options_with_spot["spot_price"].isna()]
        ["ticker"]
        .unique()
    )
    print(f"⚠ {nan_count:,} NaNs found in spot_price.")
    print(f"  Affected tickers: {problem_tickers}")
    print("  Check that these tickers exist in price_history.csv.")


In [ ]:
# Compute moneyness

# Moneyness = strike / spot price
# = 1.0  → at the money  (strike == spot)
# > 1.0  → out of the money for calls / in the money for puts
# < 1.0  → in the money for calls  / out of the money for puts

options_with_spot["moneyness"] = (
    options_with_spot["strike"] / options_with_spot["spot_price"]
).round(4)

print("\nMoneyness sample:")
print(
    options_with_spot[["ticker", "expiration", "contractType", 
                        "strike", "spot_price", "moneyness"]]
    .head(10)
    .to_string(index=False)
)

In [ ]:
# Quick sanity check

summary = (
    options_with_spot.groupby("ticker")
    .agg(
        contracts  = ("strike",     "count"),
        spot_price = ("spot_price", "first"),
        min_strike = ("strike",     "min"),
        max_strike = ("strike",     "max"),
        min_money  = ("moneyness",  "min"),
        max_money  = ("moneyness",  "max"),
    )
    .round(4)
)
print("\nSummary by ticker:")
print(summary)

In [ ]:
# Save merged dataset

OUTPUT_MERGED = "options_with_spot.csv"

options_with_spot.to_csv(OUTPUT_MERGED, index=False)
print(f"\nSaved → {OUTPUT_MERGED}  ({options_with_spot.shape[0]:,} rows, "
      f"{options_with_spot.shape[1]} columns)")


In [ ]:
# Reload check

reloaded = pd.read_csv(OUTPUT_MERGED, parse_dates=["expiration"])
assert reloaded.shape == options_with_spot.shape, "Reload shape mismatch!"
print(f"Reload check passed — {reloaded.shape[0]:,} rows, "
      f"{reloaded.shape[1]} columns.")
reloaded.head()

# Section 4 — Liquidity Filtering

The bundled dataset already includes liquidity-filtered options. Load it below, or run the "Optional" cells to recompute from fresh data.

In [ ]:
# Load bundled filtered options (no network needed)

from etf_iv.data import load_options_filtered

options_filtered = load_options_filtered()
print(f"Loaded bundled filtered options: {len(options_filtered):,} rows")
options_filtered.head()

### Optional: Recompute Liquidity Filtering

Skip these cells if you loaded the bundled data above.

In [ ]:
# Configuration

MIN_VOLUME        = 1
MIN_OPEN_INTEREST = 10
OUTPUT_FILTERED   = "options_filtered.csv"

In [ ]:
# Load data

options = pd.read_csv("options_with_spot.csv", parse_dates=["expiration"])
print(f"Loaded: {len(options):,} rows")

In [ ]:
# Record pre-filter counts

before_counts = options.groupby("ticker").size().rename("before")
print("Contracts before filtering:")
print(before_counts.to_string())

In [ ]:
# Apply liquidity filters

mask = (
    (options["volume"].fillna(0)       >= MIN_VOLUME) &
    (options["openInterest"].fillna(0) >= MIN_OPEN_INTEREST)
)
options_filtered = options[mask].reset_index(drop=True)

print(f"Thresholds  :  volume >= {MIN_VOLUME},  openInterest >= {MIN_OPEN_INTEREST}")
print(f"Before      : {len(options):,} contracts")
print(f"After       : {len(options_filtered):,} contracts")
print(f"Dropped     : {len(options) - len(options_filtered):,} contracts")

In [ ]:
# Per-ticker drop report

after_counts = options_filtered.groupby("ticker").size().rename("after")
report = pd.concat([before_counts, after_counts], axis=1).fillna(0).astype(int)
report["dropped"]     = report["before"] - report["after"]
report["pct_dropped"] = (report["dropped"] / report["before"] * 100).round(1)
print("Liquidity filter report (per ticker):")
print(report.to_string())

In [ ]:
# Save filtered dataset

options_filtered.to_csv(OUTPUT_FILTERED, index=False)
print(f"Saved → {OUTPUT_FILTERED}  ({options_filtered.shape[0]:,} rows, "
      f"{options_filtered.shape[1]} columns)")

In [ ]:
# Reload check

reloaded = pd.read_csv(OUTPUT_FILTERED, parse_dates=["expiration"])
assert reloaded.shape == options_filtered.shape, "Reload shape mismatch!"
print(f"Reload check passed — {reloaded.shape[0]:,} rows.")
reloaded.head()

# Section 5 — Holdings

Load the bundled top holdings below, or run the "Optional" cells to scrape current data from stockanalysis.com.

In [ ]:
# Load bundled ETF holdings (no network needed)

from etf_iv.data import load_etf_holdings

holdings = load_etf_holdings()
print(f"Loaded bundled holdings: {len(holdings):,} rows, "
      f"{holdings['etf_ticker'].nunique()} ETFs")
holdings.head()

### Optional: Scrape Fresh Holdings

Skip these cells if you loaded the bundled data above. Run them to scrape **current** holdings from stockanalysis.com.

In [ ]:
# Imports

import time
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [ ]:
# Configuration

TICKERS     = ["VOO", "QQQ", "ARKQ", "BOTZ"]
OUTPUT_CSV  = "etf_holdings.csv"
DELAY       = 1.5   # seconds between requests — polite scraping

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

def build_url(ticker: str) -> str:
    return f"https://stockanalysis.com/etf/{ticker.lower()}/holdings/"


In [ ]:
# Scraper function

def scrape_holdings(ticker: str) -> pd.DataFrame:
    """
    Scrape top holdings from stockanalysis.com for a given ETF ticker.

    Returns a DataFrame with columns:
        holding_ticker, company_name, weight_pct, etf_ticker

    Raises ValueError if the page or table cannot be parsed.
    """
    url = build_url(ticker)

    # --- Fetch page ---
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
    except requests.RequestException as e:
        raise ValueError(f"HTTP request failed for {ticker}: {e}") from e

    soup = BeautifulSoup(resp.text, "lxml")

    # --- Find the holdings table ---
    table = soup.find("table")
    if table is None:
        raise ValueError(
            f"No table found for {ticker}. "
            "Page structure may have changed."
        )

    # --- Parse rows ---
    rows = []
    for tr in table.find("tbody").find_all("tr"):
        cells = tr.find_all("td")
        if len(cells) < 4:
            continue

        # Column order: No. | Symbol | Name | % Weight | Shares
        symbol  = cells[1].get_text(strip=True)
        name    = cells[2].get_text(strip=True)
        weight  = cells[3].get_text(strip=True)

        # Clean weight — strip "%" and convert to float
        try:
            weight_float = float(weight.replace("%", "").strip())
        except ValueError:
            weight_float = float("nan")

        rows.append({
            "holding_ticker": symbol,
            "company_name"  : name,
            "weight_pct"    : weight_float,
            "etf_ticker"    : ticker.upper(),
        })

    if not rows:
        raise ValueError(f"Table found but no rows parsed for {ticker}.")

    return pd.DataFrame(rows)


In [ ]:
# Fetch holdings for all tickers

frames = []

for i, ticker in enumerate(TICKERS):
    print(f"Scraping holdings: {ticker} ...", end=" ")
    try:
        df = scrape_holdings(ticker)
        frames.append(df)
        print(f"✓  {len(df)} holdings")
    except ValueError as e:
        print(f"✗  Skipped — {e}")

    # Polite delay between requests (skip after last ticker)
    if i < len(TICKERS) - 1:
        time.sleep(DELAY)

if not frames:
    raise RuntimeError("No holdings data collected.")

holdings = pd.concat(frames, ignore_index=True)
print(f"\nTotal rows: {len(holdings):,}")

In [ ]:
# Verify and preview

# Check for unexpected NaNs
nan_weights = holdings["weight_pct"].isna().sum()
if nan_weights == 0:
    print("✓ No NaNs in weight_pct")
else:
    print(f"⚠ {nan_weights} NaN weights — inspect rows below:")
    print(holdings[holdings["weight_pct"].isna()])

# Coverage check — confirm all 4 ETFs scraped
scraped = holdings["etf_ticker"].unique()
missing = set(TICKERS) - set(scraped)
if missing:
    print(f"⚠ Missing ETFs: {missing}")
else:
    print(f"✓ All ETFs present: {sorted(scraped)}")

# Preview top 5 holdings per ETF
print("\nTop 5 holdings per ETF:")
print(
    holdings.groupby("etf_ticker")
    .head(5)
    .to_string(index=False)
)

In [ ]:
# Summary stats

summary = (
    holdings.groupby("etf_ticker")
    .agg(
        num_holdings    = ("holding_ticker", "count"),
        top10_weight    = ("weight_pct",     lambda x: x.head(10).sum()),
        max_weight      = ("weight_pct",     "max"),
    )
    .round(2)
)
print(summary)

In [ ]:
# Save to CSV

holdings.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved → {OUTPUT_CSV}  ({holdings.shape[0]:,} rows, "
      f"{holdings.shape[1]} columns)")

In [ ]:
# Reload check

reloaded = pd.read_csv(OUTPUT_CSV)
assert reloaded.shape == holdings.shape, "Reload shape mismatch!"
print(f"Reload check passed — {reloaded.shape[0]:,} rows.")
reloaded.head()

# Section 6 — Earnings Dates

Load the bundled earnings dates below, or run the "Optional" cells to scrape current dates from stockanalysis.com.

In [ ]:
# Load bundled earnings dates (no network needed)

from etf_iv.data import load_earnings_dates

earnings_df = load_earnings_dates()
print(f"Loaded bundled earnings dates: {len(earnings_df):,} rows, "
      f"{earnings_df['earnings_date'].notna().sum()} with dates")
earnings_df.head()

### Optional: Scrape Fresh Earnings Dates

Skip these cells if you loaded the bundled data above. Run them to scrape **current** earnings dates from stockanalysis.com.

In [ ]:
# Imports

import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [ ]:
# Configuration

OUTPUT_CSV = "earnings_dates.csv"
DELAY      = 1.5   # seconds between requests — polite scraping

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

def build_url(ticker: str) -> str:
    return f"https://stockanalysis.com/stocks/{ticker.lower()}/statistics/"

In [ ]:
# Load holdings from previous section

holdings = pd.read_csv("etf_holdings.csv")

print(f"Loaded {len(holdings):,} holdings across "
      f"{holdings['etf_ticker'].nunique()} ETFs")

# Deduplicate — same stock can appear in multiple ETFs (e.g. AAPL in VOO + QQQ)
# We only need to scrape each holding_ticker once
unique_holdings = (
    holdings[["holding_ticker", "etf_ticker"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

all_tickers = unique_holdings["holding_ticker"].unique()

# Skip tickers with exchange prefixes (e.g. "TLV: ESLT") — these produce
# invalid stockanalysis.com URLs and always return 404.
unique_tickers = [t for t in all_tickers if ":" not in t and "/" not in t]
skipped = len(all_tickers) - len(unique_tickers)

print(f"Unique holding tickers          : {len(all_tickers)}")
if skipped:
    print(f"Skipped (exchange-prefixed)      : {skipped}")
print(f"Tickers to scrape               : {len(unique_tickers)}")


In [ ]:
# Scraper function

def scrape_earnings_date(ticker: str) -> str | None:
    """
    Scrape the next estimated earnings date for a stock from
    stockanalysis.com/stocks/{ticker}/statistics/.

    Returns the earnings date as a string (e.g. 'Apr 30, 2026'),
    or None if not found or the request fails.
    """
    url = build_url(ticker)

    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"  ⚠ HTTP error for {ticker}: {e}")
        return None

    soup = BeautifulSoup(resp.text, "lxml")

    # Find the "Important Dates" section — look for a table row
    # where the first cell says "Earnings Date"
    for row in soup.find_all("tr"):
        cells = row.find_all("td")
        if len(cells) == 2 and "Earnings Date" in cells[0].get_text():
            return cells[1].get_text(strip=True)

    return None   # No earnings date found

In [ ]:
# Scrape earnings dates for all unique holdings

earnings_map = {}   # ticker -> earnings_date string or None

for i, ticker in enumerate(unique_tickers):
    date = scrape_earnings_date(ticker)
    earnings_map[ticker] = date

    status = f"✓  {date}" if date else "—  No date found"
    print(f"[{i+1:>3}/{len(unique_tickers)}] {ticker:<8} {status}")

    # Polite delay between requests
    if i < len(unique_tickers) - 1:
        time.sleep(DELAY)

In [ ]:
# Build earnings DataFrame

# Join earnings dates back onto the full holdings table
# so each row has both etf_ticker and earnings_date

earnings_df = (
    unique_holdings
    .copy()
    .assign(earnings_date=lambda df: df["holding_ticker"].map(earnings_map))
)

# Parse to datetime where possible — keep NaT for missing dates
earnings_df["earnings_date"] = pd.to_datetime(
    earnings_df["earnings_date"], errors="coerce"
)

print(f"\nTotal rows       : {len(earnings_df):,}")
print(f"With date        : {earnings_df['earnings_date'].notna().sum():,}")
print(f"Missing date     : {earnings_df['earnings_date'].isna().sum():,}")


In [ ]:
# Preview results

print("\nUpcoming earnings (soonest first):")
print(
    earnings_df
    .dropna(subset=["earnings_date"])
    .sort_values("earnings_date")
    .head(20)
    .to_string(index=False)
)

print("\nHoldings with no earnings date:")
print(
    earnings_df[earnings_df["earnings_date"].isna()]
    .to_string(index=False)
)

In [ ]:
# Save to CSV

earnings_df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved → {OUTPUT_CSV}  ({earnings_df.shape[0]:,} rows, "
      f"{earnings_df.shape[1]} columns)")


# Section 7 — ETF News Sentiment Analysis

Load the bundled sentiment scores below. To get **fresh** sentiment, set your `ANTHROPIC_API_KEY` environment variable and run the "Optional" collection cells instead.

In [ ]:
# Load bundled sentiment scores (no API key needed)

from etf_iv.data import load_etf_sentiment

etf_sentiment = load_etf_sentiment()
print(f"Loaded bundled sentiment: {len(etf_sentiment)} ETFs")
etf_sentiment

### Optional: Collect Fresh Sentiment Scores

Skip these cells if you loaded the bundled data above. To run them, set your `ANTHROPIC_API_KEY` environment variable first:

```bash
export ANTHROPIC_API_KEY="sk-ant-..."
```

In [ ]:
# Imports

import anthropic
import yfinance as yf
import pandas as pd

In [ ]:
# Configuration

TICKERS      = ["VOO", "QQQ", "ARKQ", "BOTZ"]
MODEL        = "claude-sonnet-4-5-20250514"
MAX_ARTICLES = 10
OUTPUT_CSV   = "etf_sentiment.csv"

SENTIMENT_TOOL = {
    "name": "record_sentiment",
    "description": "Record the overall sentiment score for ETF news headlines.",
    "input_schema": {
        "type": "object",
        "properties": {
            "sentiment_score": {
                "type": "integer",
                "description": (
                    "Overall sentiment: "
                    "1 = Very Negative, 2 = Negative, 3 = Neutral, "
                    "4 = Positive, 5 = Very Positive"
                ),
                "minimum": 1,
                "maximum": 5,
            }
        },
        "required": ["sentiment_score"],
    },
}

SCORE_LABELS = {
    1: "Very Negative",
    2: "Negative",
    3: "Neutral",
    4: "Positive",
    5: "Very Positive",
}

client = anthropic.Anthropic()
print(f"Model          : {MODEL}")
print(f"Max articles   : {MAX_ARTICLES}")
print(f"Output         : {OUTPUT_CSV}")

In [ ]:
# Helper — fetch news headlines via yfinance

def get_etf_headlines(ticker: str, max_articles: int = MAX_ARTICLES) -> list[str]:
    """Return up to *max_articles* recent news headlines for *ticker*."""
    t = yf.Ticker(ticker)
    articles = t.news or []
    headlines = []
    for article in articles[:max_articles]:
        title = article.get("title") or article.get("headline", "")
        if title:
            headlines.append(title)
    return headlines

In [ ]:
# Helper — score headlines via Anthropic structured output

def get_sentiment_score(ticker: str, headlines: list[str]) -> int:
    """Call Claude Sonnet with forced tool use and return a 1-5 sentiment score."""
    prompt = (
        f"You are a financial analyst. Given these recent news headlines about "
        f"the {ticker} ETF, rate the overall market sentiment on a scale of "
        f"1 (very negative) to 5 (very positive). Use the record_sentiment "
        f"tool to return your score.\n\n"
        + "\n".join(f"- {h}" for h in headlines)
    )

    response = client.messages.create(
        model=MODEL,
        max_tokens=64,
        tools=[SENTIMENT_TOOL],
        tool_choice={"type": "tool", "name": "record_sentiment"},
        messages=[{"role": "user", "content": prompt}],
    )

    for block in response.content:
        if block.type == "tool_use":
            return int(block.input["sentiment_score"])

    return 3  # fallback: neutral

In [ ]:
# Fetch headlines and score sentiment for each ETF

results = []

for ticker in TICKERS:
    print(f"Processing {ticker} ...", end=" ")

    headlines = get_etf_headlines(ticker)
    if not headlines:
        print("⚠ No headlines found — defaulting to Neutral.")
        results.append({
            "etf_ticker":      ticker,
            "num_articles":    0,
            "sentiment_score": 3,
        })
        continue

    score = get_sentiment_score(ticker, headlines)
    results.append({
        "etf_ticker":      ticker,
        "num_articles":    len(headlines),
        "sentiment_score": score,
    })
    print(f"✓  {len(headlines)} headlines → score {score} ({SCORE_LABELS[score]})")

print(f"\nScored all {len(TICKERS)} ETFs.")

In [ ]:
# Build sentiment DataFrame

etf_sentiment = pd.DataFrame(results)
etf_sentiment["sentiment_label"] = etf_sentiment["sentiment_score"].map(SCORE_LABELS)

print("ETF Sentiment Summary:")
print(etf_sentiment.to_string(index=False))

In [ ]:
# Save to CSV

etf_sentiment.to_csv(OUTPUT_CSV, index=False)
print(f"Saved → {OUTPUT_CSV}  ({etf_sentiment.shape[0]} rows, "
      f"{etf_sentiment.shape[1]} columns)")

In [ ]:
# Reload check

reloaded = pd.read_csv(OUTPUT_CSV)
assert reloaded.shape == etf_sentiment.shape, "Reload shape mismatch!"
print(f"Reload check passed — {reloaded.shape[0]} rows.")
reloaded

In [ ]:
# Merge sentiment into holdings for a combined preview

from etf_iv.data import load_etf_holdings
holdings = load_etf_holdings()

holdings_with_sentiment = holdings.merge(
    etf_sentiment[["etf_ticker", "sentiment_score", "sentiment_label"]],
    on="etf_ticker",
    how="left",
)

print("Holdings with ETF-level sentiment (top 5 per ETF):")
print(
    holdings_with_sentiment
    .groupby("etf_ticker")
    .head(5)
    .to_string(index=False)
)

# Section 8 — Put/Call Ratio Predictive Analysis

**Question:** Do extreme put/call ratios (PCR) predict short-term price direction for the underlying ETF?

**Methodology:**

We take a two-pronged approach, working within the constraint that yfinance provides a *current snapshot* of the options chain rather than a historical time series of PCR values.

1. **Cross-sectional analysis** — Treat the 4 ETF tickers as independent observations. Compute each ETF's aggregate PCR (volume-based and open-interest-based) and compare it against subsequent 1-day and 5-day price returns.

2. **Expiration-bucketed analysis** — For each ETF, compute a separate PCR for each option expiration date. This gives many more observations per ticker and allows us to test whether PCR levels at different time horizons correlate with expected returns to that expiration.

We then apply simple threshold-based signals (`PCR > 1.5` = bearish, `PCR < 0.5` = bullish) and visualize the results with dual-axis time-series plots, bar charts, and scatter plots.

In [ ]:
# Compute put/call ratios from filtered options data

import pandas as pd
import numpy as np
from etf_iv.data import load_options_filtered

opts = load_options_filtered()

# --- Aggregate PCR by ticker (volume-based and OI-based) ---

agg = (
    opts.groupby(["ticker", "contractType"])[["volume", "openInterest"]]
    .sum()
    .unstack("contractType")
)
agg.columns = ["_".join(c) for c in agg.columns]

pcr_ticker = pd.DataFrame({
    "pcr_volume": agg["volume_put"] / agg["volume_call"],
    "pcr_oi":     agg["openInterest_put"] / agg["openInterest_call"],
    "put_volume": agg["volume_put"],
    "call_volume": agg["volume_call"],
    "put_oi": agg["openInterest_put"],
    "call_oi": agg["openInterest_call"],
}).round(4)

print("=== Aggregate Put/Call Ratios by Ticker ===\n")
print(pcr_ticker.to_string())

# --- PCR by ticker × expiration date ---

agg_exp = (
    opts.groupby(["ticker", "expiration", "contractType"])[["volume", "openInterest"]]
    .sum()
    .unstack("contractType")
    .fillna(0)
)
agg_exp.columns = ["_".join(c) for c in agg_exp.columns]

pcr_by_expiry = pd.DataFrame({
    "pcr_volume": agg_exp["volume_put"] / agg_exp["volume_call"].replace(0, np.nan),
    "pcr_oi":     agg_exp["openInterest_put"] / agg_exp["openInterest_call"].replace(0, np.nan),
}).reset_index()

# Bucket expirations: near (≤30d), mid (31-90d), far (>90d)
today = pd.Timestamp.today().normalize()
pcr_by_expiry["days_to_exp"] = (pcr_by_expiry["expiration"] - today).dt.days
pcr_by_expiry["horizon"] = pd.cut(
    pcr_by_expiry["days_to_exp"],
    bins=[-np.inf, 30, 90, np.inf],
    labels=["near (≤30d)", "mid (31-90d)", "far (>90d)"]
)

print("\n=== PCR by Expiration Horizon (mean across expirations) ===\n")
horizon_summary = (
    pcr_by_expiry.groupby(["ticker", "horizon"])[["pcr_volume", "pcr_oi"]]
    .mean()
    .round(4)
)
print(horizon_summary.to_string())
print(f"\nTotal ticker × expiration observations: {len(pcr_by_expiry)}")

In [ ]:
# Compute forward returns from price history

from etf_iv.data import load_price_history
prices = load_price_history()
prices = prices.sort_values(["ticker", "Date"]).reset_index(drop=True)

# Forward returns: how much did the price move *after* each day?
# shift(-1) aligns tomorrow's return with today's row
prices["fwd_1d"] = prices.groupby("ticker")["Close"].pct_change(1).shift(-1)
prices["fwd_5d"] = prices.groupby("ticker")["Close"].pct_change(5).shift(-5)

# Latest available forward returns per ticker (most recent date with valid data)
latest_returns = (
    prices.dropna(subset=["fwd_1d"])
    .sort_values("Date")
    .groupby("ticker")
    .last()[["Date", "Close", "fwd_1d", "fwd_5d"]]
    .rename(columns={"Date": "return_date", "Close": "close_price"})
)

print("=== Most Recent Forward Returns per Ticker ===\n")
print(latest_returns.round(4).to_string())

# Also build a return-to-expiry lookup: for each future date in price history,
# compute the return from today's close to that date's close
spot = prices.groupby("ticker").last()[["Date", "Close"]].rename(
    columns={"Date": "snapshot_date", "Close": "snapshot_close"}
)

print(f"\nSnapshot dates used:")
for ticker, row in spot.iterrows():
    print(f"  {ticker}: {row['snapshot_date'].date()}  Close = ${row['snapshot_close']:.2f}")

In [ ]:
# Cross-sectional correlation and threshold-based signals

from scipy import stats

# Merge PCR with forward returns on ticker
cross = pcr_ticker[["pcr_volume", "pcr_oi"]].join(latest_returns[["fwd_1d", "fwd_5d"]])
cross = cross.dropna()

print("=== Merged PCR × Forward Returns (Cross-Sectional) ===\n")
print(cross.round(4).to_string())

# Pearson correlations (n = number of tickers with complete data)
n = len(cross)
print(f"\n=== Pearson Correlations (n = {n}) ===\n")

for pcr_col in ["pcr_volume", "pcr_oi"]:
    for ret_col in ["fwd_1d", "fwd_5d"]:
        r, p = stats.pearsonr(cross[pcr_col], cross[ret_col])
        sig = "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""
        print(f"  {pcr_col:12s} vs {ret_col:6s}:  r = {r:+.4f},  p = {p:.4f} {sig}")

print(f"\n  Note: With only {n} observations, correlations are illustrative,")
print(f"  not statistically robust. See expiration-bucketed analysis below.")

# Threshold-based signals
def classify_signal(pcr):
    if pcr > 1.5:
        return "Bearish"
    elif pcr < 0.5:
        return "Bullish"
    else:
        return "Neutral"

cross["signal_volume"] = cross["pcr_volume"].apply(classify_signal)
cross["signal_oi"]     = cross["pcr_oi"].apply(classify_signal)

print("\n=== Threshold-Based Signals ===")
print("  Rules: PCR > 1.5 → Bearish | PCR < 0.5 → Bullish | else → Neutral\n")

signal_table = cross[["pcr_volume", "signal_volume", "pcr_oi", "signal_oi",
                       "fwd_1d", "fwd_5d"]].copy()
signal_table["fwd_1d"] = (signal_table["fwd_1d"] * 100).round(2).astype(str) + "%"
signal_table["fwd_5d"] = (signal_table["fwd_5d"] * 100).round(2).astype(str) + "%"
signal_table.columns = ["PCR (Vol)", "Signal (Vol)", "PCR (OI)", "Signal (OI)",
                         "Fwd 1d Return", "Fwd 5d Return"]
print(signal_table.to_string())

In [ ]:
# Expiration-bucketed PCR vs implied return-to-expiry

# For each ticker × expiration, compute the expected return from
# the snapshot close to the price at (or nearest to) the expiration date.

# Build price lookup: for each ticker, map dates to closes
price_lookup = prices.set_index(["ticker", "Date"])["Close"].to_dict()

def get_nearest_close(ticker, target_date, prices_df):
    """Find the closing price on or nearest before target_date."""
    sub = prices_df[prices_df["ticker"] == ticker].copy()
    sub = sub[sub["Date"] <= target_date].sort_values("Date")
    if sub.empty:
        return np.nan
    return sub.iloc[-1]["Close"]

# Merge snapshot close onto PCR by expiry
pcr_exp = pcr_by_expiry.merge(
    spot.reset_index()[["ticker", "snapshot_close"]], on="ticker", how="left"
)

# For expirations that fall within our price history, compute return-to-expiry
pcr_exp["expiry_close"] = pcr_exp.apply(
    lambda row: get_nearest_close(row["ticker"], row["expiration"], prices), axis=1
)
pcr_exp["return_to_expiry"] = (
    (pcr_exp["expiry_close"] - pcr_exp["snapshot_close"]) / pcr_exp["snapshot_close"]
)

# For future expirations beyond price history, return_to_expiry will be NaN
pcr_valid = pcr_exp.dropna(subset=["pcr_volume", "return_to_expiry"]).copy()
pcr_valid["log_pcr"] = np.log(pcr_valid["pcr_volume"].replace(0, np.nan))
pcr_valid = pcr_valid.dropna(subset=["log_pcr"])

print(f"=== Expiration-Bucketed Analysis ===")
print(f"Total observations (ticker × expiration): {len(pcr_by_expiry)}")
print(f"With valid PCR + return-to-expiry:        {len(pcr_valid)}\n")

if len(pcr_valid) >= 5:
    print("--- Pearson Correlations ---")
    for x_col, label in [("pcr_volume", "PCR (volume)"), ("log_pcr", "log(PCR)")]:
        r, p = stats.pearsonr(pcr_valid[x_col], pcr_valid["return_to_expiry"])
        print(f"  {label:15s} vs return_to_expiry:  r = {r:+.4f},  p = {p:.4f}")

    print("\n--- Spearman Rank Correlations ---")
    for x_col, label in [("pcr_volume", "PCR (volume)"), ("pcr_oi", "PCR (OI)")]:
        valid = pcr_valid.dropna(subset=[x_col])
        if len(valid) >= 5:
            rho, p = stats.spearmanr(valid[x_col], valid["return_to_expiry"])
            print(f"  {label:15s} vs return_to_expiry:  rho = {rho:+.4f},  p = {p:.4f}")

    print(f"\n--- Per-Ticker Breakdown ---")
    for ticker in sorted(pcr_valid["ticker"].unique()):
        sub = pcr_valid[pcr_valid["ticker"] == ticker]
        if len(sub) >= 3:
            r, p = stats.pearsonr(sub["pcr_volume"], sub["return_to_expiry"])
            print(f"  {ticker:5s} (n={len(sub):2d}):  r = {r:+.4f},  p = {p:.4f}")
        else:
            print(f"  {ticker:5s} (n={len(sub):2d}):  Too few observations")
else:
    print("⚠ Fewer than 5 valid observations — correlations unreliable.")
    print("  This is expected when most expirations are in the future.")
    print("\n  Available data:")
    print(pcr_valid[["ticker", "expiration", "pcr_volume", "return_to_expiry"]].round(4).to_string(index=False))

In [ ]:
# Visualization A: Dual-axis time-series — price history + PCR reference

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

TICKERS = sorted(pcr_ticker.index.tolist())

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=False)
axes = axes.flatten()

for i, ticker in enumerate(TICKERS):
    ax1 = axes[i]

    # Price history
    sub = prices[prices["ticker"] == ticker].sort_values("Date")
    ax1.plot(sub["Date"], sub["Close"], color="#2563eb", linewidth=1.5, label="Close Price")
    ax1.set_ylabel("Close Price ($)", color="#2563eb")
    ax1.tick_params(axis="y", labelcolor="#2563eb")
    ax1.set_title(f"{ticker}", fontsize=13, fontweight="bold")

    # PCR on secondary y-axis
    ax2 = ax1.twinx()
    pcr_val = pcr_ticker.loc[ticker, "pcr_volume"]

    ax2.axhline(y=pcr_val, color="#dc2626", linewidth=2, linestyle="--",
                label=f"PCR (volume) = {pcr_val:.2f}")

    # Threshold shading
    pcr_max = max(pcr_val * 1.5, 2.0)
    ax2.axhspan(1.5, pcr_max, alpha=0.08, color="red", label="Bearish zone (>1.5)")
    ax2.axhspan(0, 0.5, alpha=0.08, color="green", label="Bullish zone (<0.5)")
    ax2.axhline(y=1.5, color="red", linewidth=0.8, linestyle=":", alpha=0.5)
    ax2.axhline(y=0.5, color="green", linewidth=0.8, linestyle=":", alpha=0.5)

    ax2.set_ylabel("Put/Call Ratio", color="#dc2626")
    ax2.tick_params(axis="y", labelcolor="#dc2626")
    ax2.set_ylim(0, pcr_max)

    # Snapshot date annotation
    snapshot_date = sub["Date"].max()
    ax1.axvline(x=snapshot_date, color="gray", linewidth=0.8, linestyle=":",
                alpha=0.6)
    ax1.annotate("snapshot", xy=(snapshot_date, sub["Close"].min()),
                 fontsize=8, color="gray", ha="right", va="bottom")

    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
    ax1.tick_params(axis="x", rotation=30)

    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)

fig.suptitle("ETF Price History with Snapshot Put/Call Ratio", fontsize=15, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# Visualization B: Signal-labeled bar chart of PCR by ticker

signal_colors = {"Bearish": "#ef4444", "Bullish": "#22c55e", "Neutral": "#9ca3af"}

fig, (ax_vol, ax_oi) = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, pcr_col, signal_col, title in [
    (ax_vol, "pcr_volume", "signal_volume", "Volume-Based PCR"),
    (ax_oi,  "pcr_oi",     "signal_oi",     "Open Interest-Based PCR"),
]:
    data = cross[[pcr_col, signal_col]].sort_index()
    colors = [signal_colors[s] for s in data[signal_col]]

    bars = ax.bar(data.index, data[pcr_col], color=colors, edgecolor="white", width=0.6)

    # Threshold lines
    ax.axhline(y=1.5, color="#ef4444", linewidth=1.2, linestyle="--", alpha=0.7, label="Bearish (1.5)")
    ax.axhline(y=0.5, color="#22c55e", linewidth=1.2, linestyle="--", alpha=0.7, label="Bullish (0.5)")
    ax.axhline(y=1.0, color="gray",    linewidth=0.8, linestyle=":",  alpha=0.5, label="Parity (1.0)")

    # Value labels on bars
    for bar, val, sig in zip(bars, data[pcr_col], data[signal_col]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03,
                f"{val:.2f}\n({sig})", ha="center", va="bottom", fontsize=9, fontweight="bold")

    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel("Put/Call Ratio")
    ax.set_xlabel("ETF Ticker")
    ax.legend(fontsize=8, loc="upper right")

fig.suptitle("Put/Call Ratio Signals by ETF", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Visualization C: Scatter plot — PCR vs return-to-expiry by ticker × expiration

import seaborn as sns

ticker_colors = {"ARKQ": "#ef4444", "BOTZ": "#f59e0b", "QQQ": "#3b82f6", "VOO": "#22c55e"}

fig, ax = plt.subplots(figsize=(10, 7))

if len(pcr_valid) >= 3:
    for ticker in sorted(pcr_valid["ticker"].unique()):
        sub = pcr_valid[pcr_valid["ticker"] == ticker]
        ax.scatter(sub["pcr_volume"], sub["return_to_expiry"] * 100,
                   label=ticker, color=ticker_colors.get(ticker, "gray"),
                   s=60, alpha=0.7, edgecolors="white", linewidth=0.5)

    # Overall regression line
    x = pcr_valid["pcr_volume"]
    y = pcr_valid["return_to_expiry"] * 100
    slope, intercept, r, p, se = stats.linregress(x, y)

    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = slope * x_line + intercept
    ax.plot(x_line, y_line, color="#1e293b", linewidth=2, linestyle="--",
            label=f"OLS fit (r={r:+.3f}, p={p:.3f})")

    # Confidence band (95%)
    n = len(x)
    x_mean = x.mean()
    t_crit = stats.t.ppf(0.975, n - 2)
    s_res = np.sqrt(np.sum((y - (slope * x + intercept)) ** 2) / (n - 2))
    se_line = s_res * np.sqrt(1/n + (x_line - x_mean)**2 / np.sum((x - x_mean)**2))
    ax.fill_between(x_line, y_line - t_crit * se_line, y_line + t_crit * se_line,
                    alpha=0.12, color="#1e293b", label="95% confidence band")

    # Threshold zones
    ax.axvline(x=1.5, color="#ef4444", linewidth=1, linestyle=":", alpha=0.6, label="Bearish threshold")
    ax.axvline(x=0.5, color="#22c55e", linewidth=1, linestyle=":", alpha=0.6, label="Bullish threshold")

    ax.set_xlabel("Put/Call Ratio (Volume-Based)", fontsize=12)
    ax.set_ylabel("Return to Expiry (%)", fontsize=12)
    ax.set_title("PCR vs. Implied Return to Expiry\n(each point = one ticker × expiration bucket)",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=9, loc="best")
    ax.axhline(y=0, color="gray", linewidth=0.5, alpha=0.5)
    ax.grid(alpha=0.2)
else:
    ax.text(0.5, 0.5,
            "Insufficient data for scatter plot.\n\n"
            "Most option expirations are in the future,\n"
            "so return-to-expiry cannot yet be computed.\n\n"
            "Re-run after price history extends past more expiration dates.",
            ha="center", va="center", fontsize=12, transform=ax.transAxes,
            bbox=dict(boxstyle="round,pad=0.5", facecolor="#fef3c7", alpha=0.8))
    ax.set_title("PCR vs. Implied Return to Expiry", fontsize=13, fontweight="bold")

fig.tight_layout()
plt.show()

## Interpretation of Put/Call Ratio Analysis

### What the PCR Levels Tell Us

The aggregate put/call ratios computed above reveal meaningful differences in options market sentiment across the four ETFs. A PCR near 1.0 indicates balanced call and put activity, while values significantly above or below signal asymmetric positioning. Broad-market ETFs like VOO and QQQ tend to have higher put volumes due to institutional hedging, which naturally inflates their PCR regardless of directional sentiment. Thematic ETFs like ARKQ and BOTZ, with smaller options markets, may show more volatile PCR readings driven by a smaller number of participants.

### Correlation with Forward Returns

The cross-sectional analysis (n = 4 tickers) provides directional intuition but lacks the statistical power needed for robust inference. Even with the expiration-bucketed approach — which increases the number of observations by treating each expiration date as a separate signal — the correlations should be interpreted cautiously. In traditional contrarian theory, an *extremely high* PCR (heavy put buying) is interpreted as a *bullish* signal, since it suggests excessive pessimism ripe for a reversal. If the Pearson and Spearman correlations show a negative relationship between PCR and forward returns, that would support a momentum (not contrarian) interpretation: high put/call ratios coincide with continued weakness.

### Threshold Signal Assessment

The simple threshold rules (PCR > 1.5 = bearish, PCR < 0.5 = bullish) are a common heuristic used in retail options analysis. When applied to our four ETFs, the signals should be evaluated in the context of each ETF's typical PCR range. A PCR of 1.5 for a heavily-hedged index ETF like VOO is structurally normal and does not necessarily indicate a bearish outlook, whereas the same reading for a smaller thematic ETF would be more notable.

### Limitations

This analysis has several important constraints to keep in mind:
- **Snapshot data:** yfinance provides a single current options snapshot, not a historical PCR time series. We cannot observe how PCR evolved *before* past price moves.
- **Small cross-sectional n:** With only 4 ETFs, cross-sectional correlations are illustrative, not statistically conclusive.
- **Survivorship in expirations:** Many option expirations are in the future, so return-to-expiry can only be computed for near-term expirations that fall within the 6-month price history window.
- **Confounding factors:** PCR levels are influenced by institutional hedging, market-making activity, and structural flows that may not reflect directional views.
- **No causal claim:** Correlation between PCR and returns does not establish that extreme PCR *causes* price moves in either direction.

# Section 9 — IV Skew vs. Earnings Proximity Analysis

**Question:** Does the implied-volatility skew of an ETF widen when its top holdings have upcoming earnings?

**IV Skew Definition:**

*IV Skew = median(OTM Put IV) − median(ATM Put IV)*

where OTM puts have moneyness in [0.85, 0.97) and ATM puts have moneyness in [0.97, 1.03]. We restrict to options with 15–60 days to expiration to capture the earnings-window premium while avoiding ultra-short-dated noise.

**Methodology:**

Since we have a single options snapshot (not a daily time series), we perform a **cross-sectional comparison** across the four ETFs. For each ETF we compute:

1. The IV skew from filtered options data
2. A **weighted days-to-earnings** metric based on the ETF's top-10 holdings
3. The fraction of top holdings reporting within 30 and 60 days

We then test whether ETFs whose holdings have nearer earnings also exhibit steeper volatility skews (the "earnings fear" hypothesis).

In [ ]:
# Load datasets from earlier sections

import pandas as pd
import numpy as np
from etf_iv.data import load_options_filtered, load_etf_holdings, load_earnings_dates

options = load_options_filtered()
holdings = load_etf_holdings()
earnings = load_earnings_dates()

today = pd.Timestamp.today().normalize()

# Compute days-to-expiration for every option contract
options["dte"] = (options["expiration"] - today).dt.days

print(f"Options  : {len(options):,} rows  ({options['ticker'].nunique()} ETFs)")
print(f"Holdings : {len(holdings):,} rows  ({holdings['etf_ticker'].nunique()} ETFs)")
print(f"Earnings : {len(earnings):,} rows  ({earnings['earnings_date'].notna().sum()} with dates)")
print(f"Snapshot : {today.date()}")
print(f"\nDTE range: {options['dte'].min()} – {options['dte'].max()} days")

In [ ]:
# Compute IV skew per ETF
# IV Skew = median(OTM put IV) − median(ATM put IV)
# ATM: 0.97 ≤ moneyness ≤ 1.03   |   OTM: 0.85 ≤ moneyness < 0.97
# Restricted to puts with 15–60 DTE

DTE_LO, DTE_HI = 15, 60
ATM_LO, ATM_HI = 0.97, 1.03
OTM_LO, OTM_HI = 0.85, 0.97

puts = options[
    (options["contractType"] == "put") &
    (options["dte"].between(DTE_LO, DTE_HI))
].copy()

puts["bucket"] = np.where(
    puts["moneyness"].between(ATM_LO, ATM_HI), "ATM",
    np.where(puts["moneyness"].between(OTM_LO, OTM_HI, inclusive="left"), "OTM", "other")
)
puts = puts[puts["bucket"].isin(["ATM", "OTM"])]

skew_rows = []
for ticker in sorted(options["ticker"].unique()):
    t_puts = puts[puts["ticker"] == ticker]
    atm = t_puts[t_puts["bucket"] == "ATM"]["impliedVolatility"]
    otm = t_puts[t_puts["bucket"] == "OTM"]["impliedVolatility"]

    atm_iv = atm.median() if len(atm) > 0 else np.nan
    otm_iv = otm.median() if len(otm) > 0 else np.nan
    iv_skew = otm_iv - atm_iv if pd.notna(atm_iv) and pd.notna(otm_iv) else np.nan

    skew_rows.append({
        "ticker": ticker,
        "atm_iv": atm_iv,
        "otm_iv": otm_iv,
        "iv_skew": iv_skew,
        "n_atm": len(atm),
        "n_otm": len(otm),
    })

skew_df = pd.DataFrame(skew_rows).set_index("ticker")

print("=== IV Skew by ETF ===")
print(f"  DTE window   : {DTE_LO}–{DTE_HI} days")
print(f"  ATM moneyness: [{ATM_LO}, {ATM_HI}]")
print(f"  OTM moneyness: [{OTM_LO}, {OTM_HI})\n")
print(skew_df.round(4).to_string())

In [ ]:
# Compute earnings proximity metrics per ETF

# Merge holdings with earnings dates
hold_earn = holdings.merge(
    earnings[["holding_ticker", "etf_ticker", "earnings_date"]],
    on=["holding_ticker", "etf_ticker"],
    how="left",
)

hold_earn["days_to_earnings"] = (hold_earn["earnings_date"] - today).dt.days
hold_earn["days_to_earnings"] = hold_earn["days_to_earnings"].clip(lower=0)

# Use top-10 holdings by weight for each ETF
proximity_rows = []
for etf in sorted(holdings["etf_ticker"].unique()):
    sub = (
        hold_earn[hold_earn["etf_ticker"] == etf]
        .nlargest(10, "weight_pct")
        .copy()
    )
    valid = sub.dropna(subset=["days_to_earnings"])
    if valid.empty:
        proximity_rows.append({
            "ticker": etf,
            "weighted_dte_earnings": np.nan,
            "pct_within_30d": np.nan,
            "pct_within_60d": np.nan,
            "n_holdings_with_dates": 0,
        })
        continue

    w = valid["weight_pct"]
    d = valid["days_to_earnings"]
    w_norm = w / w.sum()

    proximity_rows.append({
        "ticker": etf,
        "weighted_dte_earnings": (w_norm * d).sum(),
        "pct_within_30d": (d <= 30).mean() * 100,
        "pct_within_60d": (d <= 60).mean() * 100,
        "n_holdings_with_dates": len(valid),
    })

proximity_df = pd.DataFrame(proximity_rows).set_index("ticker")

print("=== Earnings Proximity by ETF (Top-10 Holdings) ===\n")
print(proximity_df.round(2).to_string())
print(f"\n  weighted_dte_earnings : weight-normalized avg days until top holdings report")
print(f"  pct_within_30d       : % of top-10 holdings with earnings ≤ 30 days away")
print(f"  pct_within_60d       : % of top-10 holdings with earnings ≤ 60 days away")

In [ ]:
# Merge skew with earnings proximity into one analysis table

analysis = skew_df.join(proximity_df, how="outer")

print("=== Combined IV Skew × Earnings Proximity ===\n")
print(analysis.round(4).to_string())

In [ ]:
# Visualization A: IV Skew per ETF sorted by earnings proximity

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

valid = analysis.dropna(subset=["iv_skew", "weighted_dte_earnings"]).copy()
valid = valid.sort_values("weighted_dte_earnings")

def earnings_category(d):
    if d <= 30:
        return "≤ 30 days (imminent)"
    elif d <= 60:
        return "31–60 days (near)"
    else:
        return "> 60 days (distant)"

cat_colors = {
    "≤ 30 days (imminent)": "#ef4444",
    "31–60 days (near)":    "#f59e0b",
    "> 60 days (distant)":  "#22c55e",
}

valid["proximity_cat"] = valid["weighted_dte_earnings"].apply(earnings_category)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [cat_colors[c] for c in valid["proximity_cat"]]
bars = ax.bar(valid.index, valid["iv_skew"], color=colors, edgecolor="white", width=0.55)

for bar, val, dte in zip(bars, valid["iv_skew"], valid["weighted_dte_earnings"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
            f"{val:.4f}\n({dte:.0f}d)", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.axhline(y=0, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
ax.set_xlabel("ETF Ticker (sorted by weighted days-to-earnings)", fontsize=12)
ax.set_ylabel("IV Skew  (OTM Put IV − ATM Put IV)", fontsize=12)
ax.set_title("Implied Volatility Skew by ETF\nSorted by Earnings Proximity of Top Holdings",
             fontsize=14, fontweight="bold")

handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in cat_colors.values()]
ax.legend(handles, cat_colors.keys(), title="Earnings Proximity", fontsize=9, loc="best")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()

In [ ]:
# Visualization B: Scatter — IV Skew vs Weighted DTE Earnings with trend line

fig, ax = plt.subplots(figsize=(9, 7))

ticker_colors = {"ARKQ": "#ef4444", "BOTZ": "#f59e0b", "QQQ": "#3b82f6", "VOO": "#22c55e"}

for ticker in valid.index:
    ax.scatter(
        valid.loc[ticker, "weighted_dte_earnings"],
        valid.loc[ticker, "iv_skew"],
        color=ticker_colors.get(ticker, "gray"),
        s=160, zorder=5, edgecolors="white", linewidth=1.5,
    )
    ax.annotate(
        ticker,
        (valid.loc[ticker, "weighted_dte_earnings"], valid.loc[ticker, "iv_skew"]),
        textcoords="offset points", xytext=(10, 8),
        fontsize=12, fontweight="bold", color=ticker_colors.get(ticker, "gray"),
    )

if len(valid) >= 3:
    x = valid["weighted_dte_earnings"].values
    y = valid["iv_skew"].values
    slope, intercept, r, p, se = stats.linregress(x, y)
    x_line = np.linspace(x.min() - 5, x.max() + 5, 100)
    y_line = slope * x_line + intercept
    ax.plot(x_line, y_line, color="#1e293b", linewidth=2, linestyle="--",
            label=f"OLS: skew = {slope:.6f} * dte + {intercept:.4f}\nr = {r:+.3f}, p = {p:.3f}")
    ax.legend(fontsize=10, loc="best")

ax.set_xlabel("Weighted Days-to-Earnings (top-10 holdings)", fontsize=12)
ax.set_ylabel("IV Skew  (OTM Put IV − ATM Put IV)", fontsize=12)
ax.set_title("IV Skew vs. Earnings Proximity\n(negative slope ⇒ nearer earnings → steeper skew)",
             fontsize=13, fontweight="bold")
ax.axhline(y=0, color="gray", linewidth=0.5, alpha=0.4)
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

In [ ]:
# Visualization C: Side-by-side ATM IV vs OTM IV per ETF

valid_sorted = valid.sort_values("weighted_dte_earnings")
tickers = valid_sorted.index.tolist()
x_pos = np.arange(len(tickers))
bar_w = 0.32

fig, ax = plt.subplots(figsize=(10, 6))

bars_atm = ax.bar(x_pos - bar_w / 2, valid_sorted["atm_iv"], bar_w,
                  label="ATM Put IV", color="#3b82f6", edgecolor="white")
bars_otm = ax.bar(x_pos + bar_w / 2, valid_sorted["otm_iv"], bar_w,
                  label="OTM Put IV", color="#ef4444", edgecolor="white")

for b in bars_atm:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.002,
            f"{b.get_height():.3f}", ha="center", va="bottom", fontsize=9)
for b in bars_otm:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.002,
            f"{b.get_height():.3f}", ha="center", va="bottom", fontsize=9)

# Annotate the gap (skew) between each pair
for i, ticker in enumerate(tickers):
    atm_v = valid_sorted.loc[ticker, "atm_iv"]
    otm_v = valid_sorted.loc[ticker, "otm_iv"]
    gap_y = max(atm_v, otm_v) + 0.015
    ax.annotate(
        f"skew = {otm_v - atm_v:+.4f}",
        xy=(i, gap_y), ha="center", fontsize=9, fontstyle="italic", color="#6b7280",
    )

ax.set_xticks(x_pos)
ax.set_xticklabels([f"{t}\n({valid_sorted.loc[t, 'weighted_dte_earnings']:.0f}d)" for t in tickers])
ax.set_xlabel("ETF (weighted days-to-earnings)", fontsize=12)
ax.set_ylabel("Implied Volatility", fontsize=12)
ax.set_title("ATM vs OTM Put Implied Volatility by ETF\n(gap = IV skew; sorted by earnings proximity)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()

In [ ]:
# Summary table

summary_table = analysis[["atm_iv", "otm_iv", "iv_skew", "n_atm", "n_otm",
                           "weighted_dte_earnings", "pct_within_30d", "pct_within_60d",
                           "n_holdings_with_dates"]].copy()

summary_table = summary_table.rename(columns={
    "atm_iv": "ATM IV",
    "otm_iv": "OTM IV",
    "iv_skew": "IV Skew",
    "n_atm": "# ATM Contracts",
    "n_otm": "# OTM Contracts",
    "weighted_dte_earnings": "Wtd DTE Earnings",
    "pct_within_30d": "% ≤ 30d",
    "pct_within_60d": "% ≤ 60d",
    "n_holdings_with_dates": "Holdings w/ Dates",
})

styled = (
    summary_table.style
    .format({
        "ATM IV": "{:.4f}",
        "OTM IV": "{:.4f}",
        "IV Skew": "{:+.4f}",
        "# ATM Contracts": "{:.0f}",
        "# OTM Contracts": "{:.0f}",
        "Wtd DTE Earnings": "{:.1f}",
        "% ≤ 30d": "{:.0f}%",
        "% ≤ 60d": "{:.0f}%",
        "Holdings w/ Dates": "{:.0f}",
    }, na_rep="—")
    .background_gradient(subset=["IV Skew"], cmap="RdYlGn_r")
    .background_gradient(subset=["Wtd DTE Earnings"], cmap="RdYlGn")
    .set_caption("IV Skew vs. Earnings Proximity — Summary by ETF")
)

styled

## Interpretation of IV Skew vs. Earnings Proximity

### What IV Skew Measures

Implied-volatility skew captures the market's asymmetric pricing of downside risk. When OTM put IV exceeds ATM put IV (positive skew), option traders are paying a premium for crash protection — reflecting greater fear of sharp declines. In the context of ETFs, this fear premium may intensify when the ETF's largest holdings are about to report earnings, since surprise misses from heavily-weighted constituents could drag the entire fund lower.

### Cross-Sectional Findings

The bar chart and scatter plot above compare IV skew across the four ETFs, ordered by how soon their top-10 holdings report earnings (weighted by portfolio weight). The "earnings fear" hypothesis predicts a **negative relationship**: ETFs with nearer earnings should exhibit steeper (more positive) skews. The direction and magnitude of the OLS trend line indicates whether this pattern holds in the current snapshot.

Key observations:

- **Broad-market ETFs** (VOO, QQQ) tend to carry a persistent structural skew driven by institutional hedging demand, regardless of earnings timing. This baseline skew must be considered when interpreting cross-sectional differences.
- **Thematic ETFs** (ARKQ, BOTZ) have smaller, less liquid options markets where a few large trades can shift the skew materially. Their skew readings carry more noise.
- The side-by-side ATM vs OTM chart reveals whether skew differences come from OTM put IV rising (demand for downside protection) or ATM IV falling (reduced uncertainty for near-the-money strikes), or both.

### Limitations

- **Single snapshot (n = 4):** With only four ETFs observed at one point in time, any correlation is suggestive at best. We cannot compute standard errors or confidence intervals with meaningful statistical power.
- **No time-series dimension:** A stronger test would track IV skew daily across the earnings window (e.g., 30 days before through 5 days after). This requires daily options snapshots that yfinance does not provide historically.
- **Holdings data staleness:** ETF holdings are scraped once and may not reflect intraday rebalancing. Weight percentages from stockanalysis.com are point-in-time estimates.
- **yfinance IV limitations:** Implied volatilities from yfinance are mid-market estimates that may differ from exchange-reported settlement IVs. Illiquid strikes can produce stale or noisy IV values.
- **Confounding factors:** ETFs with nearer earnings may also differ in sector composition, average market cap, or recent volatility — all of which affect skew independently of earnings proximity.
- **Moneyness bucketing:** The fixed ATM [0.97, 1.03] and OTM [0.85, 0.97) bands are standard but somewhat arbitrary. Different bin widths could shift the skew estimates, particularly for ETFs trading at very different price levels.